# 06 · Router (the adaptive part)
A router in front of retrieval decides how to handle each message:

| type | strategy |
|---|---|
| definition | retrieve k=4 |
| conceptual | retrieve k=8 |
| compare | retrieve per sub-query, merge |
| followup | rewrite into a standalone question using the chat history, then retrieve |
| chitchat | no retrieval, short canned reply |

One LLM call in JSON mode returns `{type, standalone_query, sub_queries}`. Simple rules skip that call for obvious short definition questions and greetings.

Compared against the best fixed configuration from 04/05 (dense, k=5) on the eval set, plus follow-up and chit-chat cases (`data/eval/router_cases.json`). Output: `data/eval/router_results.json`.

## 1 · Load the 05 pipeline
The retrieval, prompt, citation and support-check functions are the ones in `05_generation.ipynb`: its code cells up to its **Checks** section are run here, so both notebooks use one definition. (They move into `src/edu_rag/` after 07.)

In [1]:
import json, re, time
from pathlib import Path

import nbformat


def run_pipeline_cells(path="05_generation.ipynb"):
    nb = nbformat.read(path, as_version=4)
    for cell in nb.cells:
        if cell.cell_type == "markdown" and cell.source.lstrip().startswith("---") and "# Checks" in cell.source:
            break
        if cell.cell_type == "code":
            exec(cell.source, globals())


run_pipeline_cells()
ROUTER_CASES = EVAL_DIR / "router_cases.json"
print(f"pipeline from 05: {METHOD} | {EMBED_MODEL} | LLM {LLM} | dense floor {DENSE_FLOOR}")

f:\RP\Edu-Rag\rp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 605.52it/s]


retrieval: dense | bge-m3 | reorder=False | dense floor 0.45 | old reranker threshold 0.261 (comparison only) | 495 chunks
pipeline from 05: dense | bge-m3 | LLM qwen2.5:7b-instruct | dense floor 0.45


## 2 · Settings

In [2]:
K_BY_TYPE = {"definition": 4, "conceptual": 8, "followup": 5}
K_PER_SUBQUERY = 4
MAX_COMPARE_PASSAGES = 8
ROUTER_TYPES = ["definition", "conceptual", "compare", "followup", "chitchat"]

CHITCHAT_REPLIES = {
    "greeting": "Hi! Ask me anything about your course materials.",
    "thanks": "Glad it helped. Ask another question whenever you like.",
    "bye": "Bye! Good luck with your studies.",
    "other": "I can only help with questions about your course materials. Try asking about a topic from your slides.",
}

## 3 · Rules that skip the router call
- Greetings, thanks and goodbyes → chitchat.
- A short `what is / what are / define / meaning of X` question with no history-dependent words (it, this, that, they…) and no comparison words → definition.
Everything else goes to the LLM router.

In [3]:
GREETING = re.compile(r"^(hi+|hello|hey|good (morning|afternoon|evening)|hello there)\W*$", re.I)
THANKS = re.compile(r"^(thanks?|thank you|thx|ok(ay)?,? thanks?)\b", re.I)
BYE = re.compile(r"^(ok(ay)?\W*)?(bye|goodbye|see you|cya)\W*$", re.I)
DEFINITION = re.compile(r"^(what\s+(is|are)|what's|define|meaning of)\s+(an?\s+|the\s+)?(?P<term>[^?]+?)\??$", re.I)
REFERS_BACK = re.compile(r"\b(it|its|this|that|these|those|they|them|one|other|another)\b", re.I)
COMPARE_WORDS = re.compile(r"\b(differ|difference|vs\.?|versus|compare|compared|better than|instead of)\b", re.I)


VOCAB = {t for c in chunks for t in tokenize(c["title"])}   # titles only: slide text has examples like "Today it is cloudy"


def in_materials(term):
    """Every content word of the term occurs in some slide title ('weather today' does not). Others go to the LLM router."""
    words = [w for w in tokenize(term) if len(w) > 2]
    return bool(words) and all(w in VOCAB for w in words)


def rule_route(message, has_history):
    m = message.strip()
    for pat, kind in ((GREETING, "greeting"), (THANKS, "thanks"), (BYE, "bye")):
        if pat.search(m):
            return dict(type="chitchat", chitchat_kind=kind, standalone_query=m, sub_queries=[])
    d = DEFINITION.match(m)
    if (d and len(m.split()) <= 8 and not COMPARE_WORDS.search(m) and in_materials(d.group("term"))
            and not REFERS_BACK.search(d.group("term"))):
        return dict(type="definition", standalone_query=m, sub_queries=[])
    return None

## 4 · The LLM router (JSON mode)

In [4]:
ROUTER_PROMPT = """You route a student's message in a study assistant for their course materials.
Return JSON only: {"type": ..., "standalone_query": ..., "sub_queries": [...]}

type is one of:
- "definition": asks what a single term or concept is.
- "conceptual": asks how or why something works, steps, uses, problems, or a how-to about code.
- "compare": asks for the difference or relationship between two or more things.
- "followup": depends on the earlier conversation to make sense (uses words like it, that, this, the other, them).
- "chitchat": greetings, small talk, or anything unrelated to studying (weather, jokes, news, the assistant itself).

standalone_query: the message rewritten so it makes sense with no history (replace it/that/this with what they refer to). For chitchat repeat the message.
sub_queries: for "compare", one short standalone question per thing being compared. Otherwise [].
For a followup that is itself a comparison, still use type "followup" and also fill sub_queries."""


def history_text(history):
    lines = [f"{m['role']}: {m['content'][:300]}" for m in list(history)[-MAX_HISTORY_MESSAGES:]]
    return "\n".join(lines) if lines else "(no earlier conversation)"


def llm_route(message, history=()):
    msgs = [{"role": "system", "content": ROUTER_PROMPT},
            {"role": "user", "content": f"Earlier conversation:\n{history_text(history)}\n\nMessage: {message}"}]
    t0 = time.time()
    r = requests.post(f"{OLLAMA_URL}/api/chat", timeout=300, json={
        "model": LLM, "messages": msgs, "stream": False, "format": "json", "keep_alive": "10m",
        "options": {"num_ctx": NUM_CTX, "temperature": 0}})
    r.raise_for_status()
    raw = r.json()["message"]["content"]
    try:
        out = json.loads(raw)
        assert out.get("type") in ROUTER_TYPES
        out["standalone_query"] = str(out.get("standalone_query") or message).strip()
        out["sub_queries"] = [str(s).strip() for s in (out.get("sub_queries") or []) if str(s).strip()][:3]
        out["parse_ok"] = True
    except (ValueError, AssertionError, AttributeError):
        out = dict(type="conceptual", standalone_query=message, sub_queries=[], parse_ok=False)   # safe fallback
    out["router_seconds"] = round(time.time() - t0, 2)
    return out


def route(message, history=()):
    r = rule_route(message, bool(history))
    if r:
        return dict(r, by="rule", router_seconds=0.0, parse_ok=True)
    return dict(llm_route(message, history), by="llm")

## 5 · Retrieval per route, then the 05 answer step
`compare` retrieves for each sub-query and interleaves the results (one from each side in turn), so both sides of the comparison are in the passages. The dense floor uses the best dense score over everything retrieved.

In [5]:
def retrieve_for_route(r, roles=EVIDENCE_ROLES, sources=None):
    if r["type"] == "compare" or (r["type"] == "followup" and len(r["sub_queries"]) >= 2):
        lists, tops = [], []
        for sq in r["sub_queries"] or [r["standalone_query"]]:
            p, top, _ = retrieve(sq, k=K_PER_SUBQUERY, roles=roles, sources=sources)
            lists.append(p)
            tops.append(top)
        merged, seen = [], set()
        for i in range(max(map(len, lists), default=0)):
            for lst in lists:
                if i < len(lst) and lst[i]["chunk_id"] not in seen:
                    merged.append(lst[i])
                    seen.add(lst[i]["chunk_id"])
        return merged[:MAX_COMPARE_PASSAGES], max(tops, default=0.0)
    p, top, _ = retrieve(r["standalone_query"], k=K_BY_TYPE.get(r["type"], K), roles=roles, sources=sources)
    return p, top


def generate(question, passages, top_dense, style="concise", history=()):
    """The answer step of 05's answer(), for passages chosen by the router."""
    out = dict(abstained=False, abstain_reason=None, answer=None, sources=[], citations_ok=False, support=[], llm=None,
               llm_insufficient=False)
    if not passages or top_dense < DENSE_FLOOR:
        return dict(out, abstained=True, abstain_reason="off-topic (below dense floor)", answer=ABSTAIN_MSG)
    raw, stats = chat(build_messages(question, passages, style, history))
    out.update(llm=stats, llm_insufficient=INSUFFICIENT in raw)
    if out["llm_insufficient"]:
        return dict(out, abstained=True, abstain_reason="model said insufficient", answer=ABSTAIN_MSG)
    text, valid, invalid = parse_citations(title_citations_to_labels(raw, passages), len(passages))
    out.update(answer=text, invalid_citations=invalid, support=support_check(text, passages))
    if valid:
        out.update(citations_ok=True, sources=[source_entry(passages[n - 1], n) for n in valid])
    else:
        out["sources"] = [source_entry(c, i) for i, c in enumerate(passages[:3], start=1)]
    return out


def adaptive_answer(message, history=(), style="concise"):
    t0 = time.time()
    r = route(message, history)
    res = dict(question=message, route=r)
    if r["type"] == "chitchat":
        kind = r.get("chitchat_kind", "other")
        # No grounded answer is given, so it counts as abstained in the evaluation.
        res.update(abstained=True, abstain_reason="chitchat", answer=CHITCHAT_REPLIES[kind], sources=[], retrieved=[],
                   top_dense=None, llm=None, citations_ok=False, support=[])
    else:
        passages, top = retrieve_for_route(r)
        # The answer step sees the standalone question, so follow-ups are answered about the right thing.
        res.update(generate(r["standalone_query"], passages, top, style, history), retrieved=[c["chunk_id"] for c in passages],
                   top_dense=top)
    res["seconds"] = round(time.time() - t0, 2)
    return res


def show_adaptive(res):
    r = res["route"]
    print(f"Q: {res['question']}")
    print(f"   route: {r['type']} by {r['by']} ({r['router_seconds']} s) -> {r['standalone_query']!r}"
          + (f" sub={r['sub_queries']}" if r["sub_queries"] else ""))
    print(f"   {len(res['retrieved'])} passages, {res['seconds']} s")
    print("  ", (res["answer"] or "")[:400].replace("\n", "\n   "))
    if res["sources"]:
        print("   sources:", "; ".join(f"{s['source'][:28]} {s['pages']}" for s in res["sources"]))
    print()

---
# Checks

## 6 · A few messages end to end

In [6]:
demo = [("What is a nominal attribute?", []),
        ("What is the difference between match() and search() in the re module?", []),
        ("hi", []),
        ("What is the weather today?", []),
        ("How is it different from a list?", [{"role": "user", "content": "What is a tuple in Python?"},
                                              {"role": "assistant", "content": "A tuple is an ordered, immutable collection [S1]."}])]
for msg, hist in demo:
    show_adaptive(adaptive_answer(msg, hist))

Q: What is a nominal attribute?
   route: definition by rule (0.0 s) -> 'What is a nominal attribute?'
   4 passages, 5.95 s
   A nominal attribute refers to "relating to names" and also known as categorical, where each value represents a kind of category, code, or state. The values do not have any meaningful order [S2].
   sources: Ch 1 Data Mining Basics.pdf p. 30

Q: What is the difference between match() and search() in the re module?
   route: compare by llm (3.2 s) -> 'What is the difference between match() and search() in the re module?' sub=['What does match() do in the re module?', 'What does search() do in the re module?']
   6 passages, 10.78 s
   Match searches only at the beginning of the string, returning a match object if found [S1][S2]. Search looks for a pattern anywhere within the string [S2].
   sources: Ch 7. Regular Expressions.pd p. 4-5; Ch 7. Regular Expressions.pd p. 7

Q: hi
   route: chitchat by rule (0.0 s) -> 'hi'
   0 passages, 0.0 s
   Hi! Ask me anything 

## 7 · Rules on hand-made cases

In [7]:
for m, h in [("What is DBSCAN?", False), ("what's a lambda", False), ("What is it used for?", True), ("What is it used for?", False),
             ("What is the difference between list and tuple?", False), ("thanks!", False), ("ok bye", False),
             ("How does pruning work?", False), ("hello there", False), ("What is the weather today?", False)]:
    r = rule_route(m, h)
    print(f"{m!r:52} history={h!s:5} -> {r['type'] if r else 'LLM router'}")

'What is DBSCAN?'                                    history=False -> definition
"what's a lambda"                                    history=False -> definition
'What is it used for?'                               history=True  -> LLM router
'What is it used for?'                               history=False -> LLM router
'What is the difference between list and tuple?'     history=False -> LLM router
'thanks!'                                            history=False -> chitchat
'ok bye'                                             history=False -> chitchat
'How does pruning work?'                             history=False -> LLM router
'hello there'                                        history=False -> chitchat
'What is the weather today?'                         history=False -> LLM router


## 8 · Eval set: router vs the best fixed configuration
The fixed baseline is 05's run (dense, k=5, same abstain rule), loaded from `generation_results.jsonl`. The router is run on the same 68 questions with no history.
Retrieval is scored on the passages actually sent to the LLM: Hit (any gold page among them) and page Recall (share of gold pages covered).

In [8]:
with open(EVAL_DIR / "questions.csv", encoding="utf-8") as f:
    questions = list(csv.DictReader(f))
for q in questions:
    q["answerable"] = q["answerable"].strip().lower() == "yes"
    q["gold_pages"] = [int(p) for p in q["gold_pages"].split(";") if p.strip()]
baseline = {r["question"]: r for r in map(json.loads, (EVAL_DIR / "generation_results.jsonl").read_text(encoding="utf-8").splitlines())}


def is_relevant(c, q):
    return c["source_name"] == q["gold_source"] and any(c["page_start"] <= p <= c["page_end"] for p in q["gold_pages"])


def page_recall(chunk_ids, q):
    hit = {p for cid in chunk_ids for p in q["gold_pages"]
           if by_id[cid]["source_name"] == q["gold_source"] and by_id[cid]["page_start"] <= p <= by_id[cid]["page_end"]}
    return len(hit) / len(q["gold_pages"])


t0 = time.time()
router_runs = []
for q in questions:
    res = adaptive_answer(q["question"])
    res.update(type=q["type"], answerable=q["answerable"], difficulty=q["difficulty"])
    router_runs.append(res)
print(f"router: {len(router_runs)} questions in {time.time() - t0:.0f} s")

router: 68 questions in 591 s


In [9]:
def summarise(rows, get_ids):
    ans = [(q, r) for q, r in rows if q["answerable"]]
    una = [(q, r) for q, r in rows if not q["answerable"]]
    answered = [(q, r) for q, r in ans if not r["abstained"]]
    cites = [(q, r) for q, r in answered if r["citations_ok"]]
    return dict(
        hit=np.mean([any(is_relevant(by_id[c], q) for c in get_ids(r)) for q, r in ans]),
        recall=np.mean([page_recall(get_ids(r), q) for q, r in ans]),
        passages=np.mean([len(get_ids(r)) for q, r in ans]),
        answered=np.mean([not r["abstained"] for q, r in ans]),
        abstained_una=np.mean([r["abstained"] for q, r in una]),
        cites_gold=np.mean([any(is_relevant(by_id[s["chunk_id"]], q) for s in r["sources"]) for q, r in cites]),
        weak=np.mean([sum(s["flag"] == "weak support" for s in r["support"]) / max(1, len(r["support"])) for q, r in answered]),
        seconds=np.mean([r["seconds"] for q, r in rows]))


base_rows = [(q, baseline[q["question"]]) for q in questions]
rout_rows = list(zip(questions, router_runs))
cols = ["hit", "recall", "passages", "answered", "abstained_una", "cites_gold", "weak", "seconds"]
print(f"{'':28}" + "".join(f"{c:>14}" for c in cols))
results_table = {}
for name, rows in [("fixed dense k=5 (05)", base_rows), ("router", rout_rows)]:
    for d in (None, "easy", "hard"):
        sub = [(q, r) for q, r in rows if d is None or q["difficulty"] == d]
        s = summarise(sub, lambda r: r["retrieved"])
        results_table[f"{name} | {d or 'all'}"] = s
        print(f"{name + ' | ' + (d or 'all'):28}" + "".join(f"{s[c]:14.2f}" for c in cols))

                                       hit        recall      passages      answered abstained_una    cites_gold          weak       seconds
fixed dense k=5 (05) | all            0.92          0.88          5.00          1.00          1.00          0.90          0.12          5.77
fixed dense k=5 (05) | easy           1.00          0.96          5.00          1.00          1.00          0.97          0.14          5.86
fixed dense k=5 (05) | hard           0.82          0.77          5.00          1.00          1.00          0.82          0.09          5.65
router | all                          0.94          0.92          6.69          1.00          1.00          0.94          0.18          8.69
router | easy                         1.00          0.98          6.27          1.00          1.00          1.00          0.16          8.57
router | hard                         0.86          0.84          7.27          1.00          1.00          0.86          0.22          8.86


## 9 · Per question type

In [10]:
TYPES = ["definition", "conceptual", "code", "compare"]
print(f"{'':28}" + "".join(f"{t:>22}" for t in TYPES) + "    (hit / recall / passages)")
for name, rows in [("fixed dense k=5 (05)", base_rows), ("router", rout_rows)]:
    cells = []
    for t in TYPES:
        s = summarise([(q, r) for q, r in rows if q["type"] == t], lambda r: r["retrieved"])
        cells.append(f"{s['hit']:.2f} / {s['recall']:.2f} / {s['passages']:.1f}".rjust(22))
    print(f"{name:28}" + "".join(cells))

                                        definition            conceptual                  code               compare    (hit / recall / passages)
fixed dense k=5 (05)             1.00 / 0.97 / 5.0     0.89 / 0.86 / 5.0     0.83 / 0.83 / 5.0     1.00 / 0.85 / 5.0
router                           1.00 / 0.97 / 4.9     0.89 / 0.86 / 7.4     0.92 / 0.92 / 7.7     1.00 / 1.00 / 6.5


## 10 · Router decisions on the eval set
Eval `code` questions have no router type of their own; they are expected to go to conceptual (or definition). Unanswerable questions should not be called chitchat: they are course-like questions the materials don't cover.

In [11]:
from collections import Counter
conf = Counter((q["type"], r["route"]["type"]) for q, r in rout_rows)
print(f"{'eval type':12}" + "".join(f"{t:>12}" for t in ROUTER_TYPES))
for t in TYPES:
    print(f"{t:12}" + "".join(f"{conf[(t, rt)]:12}" for rt in ROUTER_TYPES))
by = Counter(r["route"]["by"] for r in router_runs)
print("\ndecided by:", dict(by), "| JSON parse failures:", sum(not r["route"]["parse_ok"] for r in router_runs))
llm_times = [r["route"]["router_seconds"] for r in router_runs if r["route"]["by"] == "llm"]
print(f"router call: mean {np.mean(llm_times):.2f} s, max {max(llm_times):.2f} s")
print("\ncompare questions, sub-queries:")
for q, r in rout_rows:
    if r["route"]["sub_queries"]:
        print(f"  {q['question'][:60]:60} -> {r['route']['sub_queries']}")
print("\nchanged outcome vs fixed:")
for q, r in rout_rows:
    b = baseline[q["question"]]
    if b["abstained"] != r["abstained"]:
        print(f"  [{q['difficulty']}] {'answerable' if q['answerable'] else 'unanswerable'}: fixed {'abstained' if b['abstained'] else 'answered'}, "
              f"router {'abstained' if r['abstained'] else 'answered'} (route {r['route']['type']}) {q['question']}")

eval type     definition  conceptual     compare    followup    chitchat
definition            13           4           0           0           0
conceptual             3          21           0           0           0
code                   1          15           0           0           0
compare                0           4           7           0           0

decided by: {'rule': 10, 'llm': 58} | JSON parse failures: 0
router call: mean 2.87 s, max 3.65 s

compare questions, sub-queries:
  What is the difference between a list and a tuple?           -> ['What are the main characteristics of a list?', 'What are the main characteristics of a tuple?']
  What is the difference between match() and search() in the r -> ['What does match() do in the re module?', 'What does search() do in the re module?']
  What is the difference between pop and del on a list?        -> ['How does the pop method work on a list?', 'How does the del statement work on a list?']
  How is agglomerative clusteri

## 11 · Follow-up questions
Each case has one earlier exchange. Compared: the raw follow-up text sent to retrieval (what 05 does) vs the router's standalone rewrite.

In [12]:
cases = json.loads(ROUTER_CASES.read_text(encoding="utf-8"))
fu_rows = []
for case in cases["followups"]:
    hist = [{"role": "user", "content": case["history_q"]}, {"role": "assistant", "content": case["history_a"]}]
    q = dict(gold_source=case["gold_source"], gold_pages=case["gold_pages"], answerable=True)
    raw_p, _, _ = retrieve(case["question"], k=K)
    res = adaptive_answer(case["question"], hist)
    fu_rows.append(dict(question=case["question"], route=res["route"]["type"], rewrite=res["route"]["standalone_query"],
                        raw_hit=any(is_relevant(c, q) for c in raw_p),
                        router_hit=any(is_relevant(by_id[c], q) for c in res["retrieved"]),
                        answered=not res["abstained"],
                        cites_gold=any(is_relevant(by_id[s["chunk_id"]], q) for s in res["sources"]) if res["citations_ok"] else False))
for r in fu_rows:
    print(f"{r['route']:10} raw {'HIT ' if r['raw_hit'] else 'miss'} router {'HIT ' if r['router_hit'] else 'miss'} "
          f"{'answered' if r['answered'] else 'ABSTAINED'}  {r['question'][:45]!r} -> {r['rewrite'][:70]!r}")
print(f"\nrouted as followup {sum(r['route'] == 'followup' for r in fu_rows)}/{len(fu_rows)} | gold in passages: raw text "
      f"{sum(r['raw_hit'] for r in fu_rows)}/{len(fu_rows)}, router {sum(r['router_hit'] for r in fu_rows)}/{len(fu_rows)} | "
      f"answered {sum(r['answered'] for r in fu_rows)}/{len(fu_rows)}, cites gold {sum(r['cites_gold'] for r in fu_rows)}/{len(fu_rows)}")

compare    raw HIT  router HIT  answered  'How is it different from a list?' -> 'What is the difference between a tuple and a list in Python?'
conceptual raw miss router HIT  answered  'Can you walk me through a numerical example o' -> 'Can you walk me through a numerical example of how DBSCAN works?'
compare    raw HIT  router HIT  answered  'And what does the bootstrap do differently?' -> 'How does k-fold cross-validation differ from bootstrapping?'
conceptual raw HIT  router HIT  answered  'How is it used to pick the splitting attribut' -> 'How is the Gini index used to pick the splitting attribute?'
conceptual raw HIT  router HIT  answered  'How do I then read it one line at a time?' -> 'How can I read a file line by line using the with statement in Python?'
conceptual raw miss router HIT  answered  'Can I use one to keep only some items of a li' -> 'Can I use a lambda function to keep only some items of a list in Pytho'
conceptual raw miss router miss answered  'Is there another w

## 12 · Chit-chat

In [13]:
cc_rows = []
for m in cases["chitchat"]:
    res = adaptive_answer(m)
    cc_rows.append(dict(message=m, route=res["route"]["type"], by=res["route"]["by"], retrieved=len(res["retrieved"]), answer=res["answer"]))
    print(f"{res['route']['type']:10} by {res['route']['by']:4} retrieval={len(res['retrieved'])}  {m!r:30} -> {res['answer'][:70]!r}")
print(f"\nchitchat recognised {sum(r['route'] == 'chitchat' for r in cc_rows)}/{len(cc_rows)}")

chitchat   by rule retrieval=0  'hi'                           -> 'Hi! Ask me anything about your course materials.'
chitchat   by rule retrieval=0  'hello there!'                 -> 'Hi! Ask me anything about your course materials.'
chitchat   by rule retrieval=0  'thanks, that helped a lot'    -> 'Glad it helped. Ask another question whenever you like.'
chitchat   by llm  retrieval=0  'who are you?'                 -> 'I can only help with questions about your course materials. Try asking'
chitchat   by llm  retrieval=0  'What is the weather today?'   -> 'I can only help with questions about your course materials. Try asking'
chitchat   by llm  retrieval=0  'tell me a joke'               -> 'I can only help with questions about your course materials. Try asking'
chitchat   by rule retrieval=0  'good morning'                 -> 'Hi! Ask me anything about your course materials.'
chitchat   by rule retrieval=0  'ok bye'                       -> 'Bye! Good luck with your studies.'

chitc

## 13 · Save

In [ ]:
clean = lambda d: {k: float(v) for k, v in d.items()}
(EVAL_DIR / "router_results.json").write_text(json.dumps(dict(
    k_by_type=K_BY_TYPE, k_per_subquery=K_PER_SUBQUERY, table={k: clean(v) for k, v in results_table.items()},
    decided_by=dict(by), followups=fu_rows, chitchat=cc_rows, cases_status=cases["status"]), indent=2))
with open(EVAL_DIR / "router_runs.jsonl", "w", encoding="utf-8") as f:
    for r in router_runs:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
unload_llm()
requests.post(f"{OLLAMA_URL}/api/embed", json={"model": EMBED_MODEL, "input": [], "keep_alive": 0}, timeout=60)
print("saved router_results.json, router_runs.jsonl |", requests.get(f"{OLLAMA_URL}/api/ps").json()["models"] or "no Ollama models loaded")

saved router_results.json, router_runs.jsonl | no Ollama models loaded


: 